# מפגש 13 — פתרון מורה בדוק, עם assert
פתרון מטלת Student-Lab המקורית: Train [100,200,300], Test [50,400]. אין fit על Test ואין אימון מודל חיזוי.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler,MinMaxScaler
train=pd.DataFrame({'measurement':[100.,200.,300.]})
test=pd.DataFrame({'measurement':[50.,400.]})
original_train=train.copy(deep=True)
original_test=test.copy(deep=True)
assert train.shape==(3,1) and test.shape==(2,1)

## משימה 1 — מחשבים את ארבעת פרמטרי האימון
σ עם ddof=0, לא ברירת המחדל ddof=1 של Pandas.

In [ ]:
mean_train=train['measurement'].mean()
std_train=train['measurement'].std(ddof=0)
min_train=train['measurement'].min()
max_train=train['measurement'].max()
print('Mean, std, min, max:',mean_train,std_train,min_train,max_train)
assert mean_train==200.
assert np.isclose(std_train,np.sqrt(20000/3))
assert min_train==100. and max_train==300.
assert not np.isclose(std_train,train['measurement'].std())

## משימה 2 — שתי המרות ידניות לעמודת Test
Test=50 קטן ממינימום Train; Test=400 גדול ממקסימום Train.

In [ ]:
manual_minmax=(test['measurement']-min_train)/(max_train-min_train)
manual_standard=(test['measurement']-mean_train)/std_train
print('Manual MinMax:',manual_minmax.tolist())
print('Manual Standard:',manual_standard.tolist())
assert np.allclose(manual_minmax,[-0.25,1.5])
assert np.allclose(manual_standard,[-1.8371173070873836,2.449489742783178])

## משימה 3 — Scalers שמתאימים רק ל־Train

In [ ]:
mm=MinMaxScaler().set_output(transform='pandas')
st=StandardScaler().set_output(transform='pandas')
mm.fit(train)
st.fit(train)
mm_train=mm.transform(train)
mm_test=mm.transform(test)
st_train=st.transform(train)
st_test=st.transform(test)
display(mm_train,mm_test,st_train,st_test)
assert np.allclose(mm_train['measurement'],[0.,0.5,1.])
assert np.allclose(mm_test['measurement'],[-0.25,1.5])
assert np.allclose(st_train['measurement'],[-np.sqrt(1.5),0.,np.sqrt(1.5)])
assert np.allclose(st_test['measurement'],manual_standard)
assert np.allclose(mm_test['measurement'],manual_minmax)

## משימה 4 — נימוק חוסר הדליפה, בדיקת הממוצע ב־Test

In [ ]:
assert np.allclose(mm.data_min_,[100.]) and np.allclose(mm.data_max_,[300.])
assert np.allclose(st.mean_,[200.]) and np.allclose(st.scale_,[np.sqrt(20000/3)])
assert np.isclose(st_train['measurement'].mean(),0.)
assert np.isclose(st_train['measurement'].std(ddof=0),1.)
assert not np.isclose(st_test['measurement'].mean(),0.)
assert train.equals(original_train)
assert test.equals(original_test)
print('Test z mean:',st_test['measurement'].mean(),'No refit on Test')

## משימה 5 — הסבר המחויב להגנה
fit לומד min/max או mean/std. transform מיישם סרגל זהה גם ל־Test, אף אם Test מחוץ לטווח האימון; MinMax−0.25 או 1.5 אינם שגיאה. Standard אינו הופך נתונים לנורמליים, וממוצע Test בתקנון אינו חייב להיות 0. שיטת Scaling היא החלטת תהליך ולא הוכחה לשיפור חיזוי.

## רשות — עמודה קבועה
זהו קצה שבו לנוסחאות הידניות מכנה 0. הספרייה מונעת חלוקה באפס; בדיקה זו אינה חומר חובה.

In [ ]:
same=np.array([[7.],[7.],[7.]])
mm_c=MinMaxScaler().fit(same)
st_c=StandardScaler().fit(same)
assert np.allclose(mm_c.transform(same),0.)
assert np.allclose(st_c.transform(same),0.)
assert np.allclose(mm_c.transform([[9.]]),[[2.]])
assert np.allclose(st_c.transform([[9.]]),[[2.]])
print('Constant train safely transformed')

## כרטיס הגנה למורה
התלמיד חייב להסביר מהו פרמטר אימון ומהו ערך בדיקה, מהו ddof=0, מדוע לא מריצים fit מחדש על Test, ומדוע אין עדיין מסקנה על Accuracy.